In [ ]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML, display
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import tensorflow as tf
from tensorflow import keras

np.random.seed(42)
tf.random.set_seed(42)

plt.rcParams.update({
    "figure.figsize": (12, 7),
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "axes.titleweight": "bold",
    "axes.grid": True,
    "grid.alpha": 0.20,
    "font.size": 11
})

OPT_COLORS = {
    "SGD": "#ff4d6d",
    "AdaGrad": "#00b4d8",
    "RMSProp": "#7b2cbf"
}

print("TensorFlow:", tf.__version__)

In [ ]:

X, y = make_regression(
    n_samples=260,
    n_features=1,
    n_informative=1,
    noise=10.0,
    bias=18.0,
    random_state=42
)

X = X.astype(np.float32)
y = y.astype(np.float32).reshape(-1, 1)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42
)

x_scaler = StandardScaler()
y_scaler = StandardScaler()

X_train_s = x_scaler.fit_transform(X_train).astype(np.float32)
X_test_s = x_scaler.transform(X_test).astype(np.float32)

y_train_s = y_scaler.fit_transform(y_train).astype(np.float32)
y_test_s = y_scaler.transform(y_test).astype(np.float32)

data_summary = pd.DataFrame({
    "Split": ["Train", "Test"],
    "Samples": [len(X_train_s), len(X_test_s)],
    "X mean": [X_train_s.mean(), X_test_s.mean()],
    "X std": [X_train_s.std(), X_test_s.std()],
    "y mean": [y_train_s.mean(), y_test_s.mean()],
    "y std": [y_train_s.std(), y_test_s.std()]
})

display(data_summary.round(4))

fig, ax = plt.subplots(figsize=(12, 7))
ax.scatter(X_train_s.ravel(), y_train_s.ravel(), s=55, alpha=0.70, label="Training data")
ax.scatter(X_test_s.ravel(), y_test_s.ravel(), s=75, alpha=0.90, marker="D", label="Test data")
ax.set_title("Synthetic Linear Regression Dataset")
ax.set_xlabel("Standardized Feature (x)")
ax.set_ylabel("Standardized Target (y)")
ax.legend(frameon=True)
plt.show()


In [ ]:
def build_linear_ann():
    return keras.Sequential([
        keras.layers.Input(shape=(1,)),
        keras.layers.Dense(1, activation=None, name="linear_neuron")
    ])

tf.keras.backend.clear_session()
tf.random.set_seed(123)

base_model = build_linear_ann()
_ = base_model(X_train_s[:1])

initial_kernel = np.array([[0.15]], dtype=np.float32)
initial_bias = np.array([1.80], dtype=np.float32)
initial_weights = [initial_kernel, initial_bias]

def fresh_model():
    model = build_linear_ann()
    model(X_train_s[:1])
    model.set_weights([w.copy() for w in initial_weights])
    return model

models = {
    "SGD": fresh_model(),
    "AdaGrad": fresh_model(),
    "RMSProp": fresh_model()
}

for name, model in models.items():
    print(name, "initial weights:", [w.ravel().tolist() for w in model.get_weights()])

In [ ]:

optimizer_settings = {
    "SGD": tf.keras.optimizers.SGD(learning_rate=0.08),
    "AdaGrad": tf.keras.optimizers.Adagrad(learning_rate=0.35, initial_accumulator_value=0.0),
    "RMSProp": tf.keras.optimizers.RMSprop(learning_rate=0.06, rho=0.90)
}

steps = 100
histories = {}

for name, model in models.items():
    optimizer = optimizer_settings[name]
    records = []

    for step in range(steps + 1):
        w, b = model.get_weights()
        prediction = model(X_train_s, training=False)
        loss = tf.reduce_mean(tf.square(prediction - y_train_s))

        with tf.GradientTape() as tape:
            prediction = model(X_train_s, training=True)
            step_loss = tf.reduce_mean(tf.square(prediction - y_train_s))

        grads = tape.gradient(step_loss, model.trainable_variables)

        grad_w = float(grads[0].numpy().ravel()[0])
        grad_b = float(grads[1].numpy().ravel()[0])

        if step == 0:
            records.append({
                "step": 0,
                "w": float(w.ravel()[0]),
                "b": float(b.ravel()[0]),
                "loss": float(loss.numpy()),
                "grad_w": grad_w,
                "grad_b": grad_b
            })
            continue

        old_w = float(w.ravel()[0])
        old_b = float(b.ravel()[0])

        optimizer.apply_gradients(zip(grads, model.trainable_variables))

        new_w, new_b = model.get_weights()

        records.append({
            "step": step,
            "w": float(new_w.ravel()[0]),
            "b": float(new_b.ravel()[0]),
            "loss": float(tf.reduce_mean(
                tf.square(model(X_train_s, training=False) - y_train_s)
            ).numpy()),
            "grad_w": grad_w,
            "grad_b": grad_b,
            "old_w": old_w,
            "old_b": old_b
        })

    histories[name] = pd.DataFrame(records)

display(histories["SGD"].head(6))

In [ ]:

def reconstruct_optimizer_state(history, name, rho=0.90):
    gw = history["grad_w"].to_numpy()
    gb = history["grad_b"].to_numpy()

    if name == "SGD":
        state = np.zeros((len(history), 2), dtype=float)
        effective_lr = np.full((len(history), 2), 0.08, dtype=float)

    elif name == "AdaGrad":
        acc = np.zeros((len(history), 2), dtype=float)
        for i in range(1, len(history)):
            acc[i] = acc[i-1] + np.array([gw[i], gb[i]])**2
        state = acc
        effective_lr = 0.35 / (np.sqrt(acc) + 1e-7)

    elif name == "RMSProp":
        ema = np.zeros((len(history), 2), dtype=float)
        for i in range(1, len(history)):
            g2 = np.array([gw[i], gb[i]])**2
            ema[i] = rho * ema[i-1] + (1-rho) * g2
        state = ema
        effective_lr = 0.06 / (np.sqrt(ema) + 1e-7)

    return state, effective_lr

states = {}
effective_lrs = {}

for name in histories:
    states[name], effective_lrs[name] = reconstruct_optimizer_state(histories[name], name)

print("State shapes:")
for name in states:
    print(f"{name:8s} -> state {states[name].shape}, effective LR {effective_lrs[name].shape}")

In [ ]:
def mse_at(w, b, X=X_train_s, y=y_train_s):
    pred = w * X + b
    return np.mean((pred - y)**2)

w_min, w_max = -1.4, 2.5
b_min, b_max = -2.4, 2.4

w_grid = np.linspace(w_min, w_max, 180)
b_grid = np.linspace(b_min, b_max, 180)
WG, BG = np.meshgrid(w_grid, b_grid)
ZG = np.zeros_like(WG)

for i in range(WG.shape[0]):
    ZG[i] = mse_at(WG[i], BG[i])

all_paths = np.concatenate([histories[name][["w", "b"]].to_numpy() for name in histories])
loss_max = np.percentile(ZG, 99)
loss_min = ZG.min()

fig = plt.figure(figsize=(15, 10))
ax = fig.add_subplot(111, projection="3d")

surface = ax.plot_surface(
    WG, BG, ZG,
    cmap="turbo",
    alpha=0.78,
    linewidth=0,
    antialiased=True
)

for name in histories:
    h = histories[name]
    ax.plot(
        h["w"], h["b"], h["loss"],
        linewidth=3.2,
        color=OPT_COLORS[name],
        label=name
    )
    ax.scatter(
        h["w"].iloc[0], h["b"].iloc[0], h["loss"].iloc[0],
        s=100, color=OPT_COLORS[name], edgecolor="black"
    )
    ax.scatter(
        h["w"].iloc[-1], h["b"].iloc[-1], h["loss"].iloc[-1],
        s=130, color=OPT_COLORS[name], marker="*",
        edgecolor="black"
    )

ax.set_title("3D MSE Landscape + Optimizer Trajectories", pad=18)
ax.set_xlabel("Weight w")
ax.set_ylabel("Bias b")
ax.set_zlabel("MSE Loss")
ax.set_zlim(loss_min, loss_max)
ax.view_init(elev=34, azim=-128)
fig.colorbar(surface, ax=ax, shrink=0.60, pad=0.10, label="MSE")
ax.legend()
plt.show()

In [ ]:

levels = np.linspace(np.percentile(ZG, 2), np.percentile(ZG, 97), 45)

fig, ax = plt.subplots(figsize=(14, 10))

filled = ax.contourf(
    WG, BG, ZG,
    levels=levels,
    cmap="turbo",
    alpha=0.95
)

ax.contour(
    WG, BG, ZG,
    levels=levels[::2],
    colors="white",
    linewidths=0.55,
    alpha=0.35
)

for name in histories:
    h = histories[name]
    ax.plot(
        h["w"], h["b"],
        color=OPT_COLORS[name],
        linewidth=3,
        label=name,
        marker="o",
        markevery=12,
        markersize=4
    )
    ax.scatter(
        h["w"].iloc[-1], h["b"].iloc[-1],
        s=180, color=OPT_COLORS[name],
        marker="*", edgecolor="black", zorder=5
    )

ax.scatter(
    all_paths[:, 0].mean(), all_paths[:, 1].mean(),
    s=0
)

ax.set_title("Contour Map: How SGD, AdaGrad and RMSProp Travel")
ax.set_xlabel("Weight w")
ax.set_ylabel("Bias b")
fig.colorbar(filled, ax=ax, label="MSE")
ax.legend()
plt.show()


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(17, 6))

for name in histories:
    h = histories[name]
    axes[0].plot(
        h["step"], h["loss"],
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )
    axes[1].semilogy(
        h["step"], h["loss"] + 1e-10,
        linewidth=3,
        color=OPT_COLORS[name],
        label=name
    )

axes[0].set_title("Training Loss")
axes[0].set_xlabel("Optimization step")
axes[0].set_ylabel("MSE")
axes[0].legend()

axes[1].set_title("Training Loss — Log Scale")
axes[1].set_xlabel("Optimization step")
axes[1].set_ylabel("MSE (log)")
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(19, 6))

for idx, name in enumerate(["SGD", "AdaGrad", "RMSProp"]):
    ax = axes[idx]
    lr = effective_lrs[name]

    ax.plot(
        lr[:, 0],
        linewidth=3,
        color=OPT_COLORS[name],
        label="w effective LR"
    )
    ax.plot(
        lr[:, 1],
        linewidth=3,
        linestyle="--",
        color="#22223b",
        label="b effective LR"
    )
    ax.set_title(f"{name}: Effective Learning Rate")
    ax.set_xlabel("Step")
    ax.set_ylabel("Effective LR")
    ax.legend()

plt.tight_layout()
plt.show()

In [ ]:

def contour_animation(name, interval=90):
    h = histories[name]
    path = h[["w", "b"]].to_numpy()

    fig, ax = plt.subplots(figsize=(12, 9))

    ax.contourf(
        WG, BG, ZG,
        levels=np.linspace(np.percentile(ZG, 2), np.percentile(ZG, 97), 45),
        cmap="turbo",
        alpha=0.95
    )
    ax.contour(
        WG, BG, ZG,
        levels=levels[::2],
        colors="white",
        linewidths=0.45,
        alpha=0.30
    )

    line, = ax.plot([], [], color=OPT_COLORS[name], linewidth=4)
    point, = ax.plot(
        [], [], marker="o", markersize=13,
        color=OPT_COLORS[name], markeredgecolor="black"
    )

    title = ax.set_title("")
    ax.set_xlabel("Weight w")
    ax.set_ylabel("Bias b")

    def update(frame):
        current = path[:frame+1]
        line.set_data(current[:, 0], current[:, 1])
        point.set_data([current[-1, 0]], [current[-1, 1]])

        row = h.iloc[frame]
        title.set_text(
            f"{name} | Step {int(row['step'])} | "
            f"w={row['w']:.4f}  b={row['b']:.4f}  "
            f"Loss={row['loss']:.5f}"
        )
        return line, point, title

    anim = FuncAnimation(
        fig, update,
        frames=len(path),
        interval=interval,
        blit=True,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())

display(contour_animation("SGD"))

In [ ]:
display(contour_animation("AdaGrad"))

In [ ]:
display(contour_animation("RMSProp"))

In [ ]:
def mechanics_animation(name, interval=140):
    h = histories[name]
    state = states[name]
    eff_lr = effective_lrs[name]

    fig, axes = plt.subplots(1, 3, figsize=(19, 6))

    ax0, ax1, ax2 = axes

    ax0.contourf(
        WG, BG, ZG,
        levels=np.linspace(np.percentile(ZG, 2), np.percentile(ZG, 97), 45),
        cmap="turbo",
        alpha=0.92
    )
    ax0.contour(
        WG, BG, ZG,
        levels=levels[::2],
        colors="white",
        linewidths=0.45,
        alpha=0.30
    )

    path_line, = ax0.plot([], [], color=OPT_COLORS[name], linewidth=4)
    path_point, = ax0.plot(
        [], [], marker="o", markersize=12,
        color=OPT_COLORS[name], markeredgecolor="black"
    )

    ax0.set_title("Where the optimizer moves")
    ax0.set_xlabel("w")
    ax0.set_ylabel("b")

    bar_names = ["grad w", "grad b", "state w", "state b"]
    xbar = np.arange(4)
    bars = ax1.bar(
        xbar, [0, 0, 0, 0],
        color=[OPT_COLORS[name], OPT_COLORS[name], "#f4a261", "#f4a261"],
        alpha=0.9
    )
    ax1.set_xticks(xbar)
    ax1.set_xticklabels(bar_names, rotation=25)
    ax1.set_title("Current step ingredients")
    ax1.axhline(0, color="black", linewidth=0.8)

    ax2.axis("off")
    info = ax2.text(
        0.02, 0.98, "",
        transform=ax2.transAxes,
        va="top",
        ha="left",
        family="monospace",
        fontsize=12,
        bbox=dict(boxstyle="round,pad=0.7", facecolor="white", alpha=0.92)
    )

    def update(frame):
        row = h.iloc[frame]
        path = h.iloc[:frame+1]

        path_line.set_data(path["w"], path["b"])
        path_point.set_data([row["w"]], [row["b"]])

        values = [
            row["grad_w"],
            row["grad_b"],
            state[frame, 0],
            state[frame, 1]
        ]

        for bar, value in zip(bars, values):
            bar.set_height(value)

        max_abs = max(1e-6, np.max(np.abs(values)) * 1.25)
        ax1.set_ylim(-max_abs, max_abs)

        if name == "SGD":
            state_formula = "No accumulated state"
        elif name == "AdaGrad":
            state_formula = "G += gradient²"
        else:
            state_formula = "v = ρv + (1-ρ)gradient²"

        text = (
            f"{name}\n"
            f"{'─'*29}\n"
            f"step       : {int(row['step'])}\n"
            f"loss       : {row['loss']:.6f}\n"
            f"w          : {row['w']:.6f}\n"
            f"b          : {row['b']:.6f}\n"
            f"grad_w     : {row['grad_w']:+.6f}\n"
            f"grad_b     : {row['grad_b']:+.6f}\n"
            f"state_w    : {state[frame,0]:.6f}\n"
            f"state_b    : {state[frame,1]:.6f}\n"
            f"eff_lr_w   : {eff_lr[frame,0]:.6f}\n"
            f"eff_lr_b   : {eff_lr[frame,1]:.6f}\n"
            f"state rule : {state_formula}"
        )
        info.set_text(text)

        return [path_line, path_point, *bars, info]

    anim = FuncAnimation(
        fig, update,
        frames=len(h),
        interval=interval,
        blit=False,
        repeat=True
    )

    plt.close(fig)
    return HTML(anim.to_jshtml())

display(mechanics_animation("SGD"))

In [ ]:
display(mechanics_animation("AdaGrad"))

In [ ]:
display(mechanics_animation("RMSProp"))

In [ ]:
results = []

for name, model in models.items():
    pred_s = model.predict(X_test_s, verbose=0)
    pred = y_scaler.inverse_transform(pred_s)
    y_true = y_test

    results.append({
        "Optimizer": name,
        "Final Train MSE (scaled)": histories[name]["loss"].iloc[-1],
        "Test MAE": mean_absolute_error(y_true, pred),
        "Test RMSE": np.sqrt(mean_squared_error(y_true, pred)),
        "Test R²": r2_score(y_true, pred),
        "Final w": model.get_weights()[0].ravel()[0],
        "Final b": model.get_weights()[1].ravel()[0]
    })

results_df = pd.DataFrame(results).sort_values("Test RMSE").reset_index(drop=True)
display(results_df.round(5))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

x = np.arange(len(results_df))
bars = ax.bar(
    x, results_df["Test RMSE"],
    color=[OPT_COLORS[o] for o in results_df["Optimizer"]],
    width=0.65
)

ax.set_xticks(x)
ax.set_xticklabels(results_df["Optimizer"])
ax.set_ylabel("Test RMSE")
ax.set_title("Final Test RMSE Comparison")

for bar, value in zip(bars, results_df["Test RMSE"]):
    ax.text(
        bar.get_x() + bar.get_width()/2,
        bar.get_height(),
        f"{value:.3f}",
        ha="center",
        va="bottom",
        fontweight="bold"
    )

plt.show()


In [ ]:

x_line = np.linspace(X.min(), X.max(), 300, dtype=np.float32).reshape(-1, 1)
x_line_s = x_scaler.transform(x_line).astype(np.float32)

fig, ax = plt.subplots(figsize=(13, 8))

ax.scatter(
    X_train.ravel(), y_train.ravel(),
    s=55, alpha=0.48, label="Train"
)
ax.scatter(
    X_test.ravel(), y_test.ravel(),
    s=85, alpha=0.88, marker="D", label="Test"
)

for name, model in models.items():
    pred_line_s = model.predict(x_line_s, verbose=0)
    pred_line = y_scaler.inverse_transform(pred_line_s).ravel()
    ax.plot(
        x_line.ravel(), pred_line,
        linewidth=3.2,
        color=OPT_COLORS[name],
        label=name
    )

ax.set_title("Learned Linear Function on the Original Data Scale")
ax.set_xlabel("Original x")
ax.set_ylabel("Original y")
ax.legend()
plt.show()